In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from typing import List, Dict, Set
import json
import os

class CryptoHistoryFetcher:
    def __init__(self, api_key: str, input_file: str, output_file: str = None):
        self.api_key = api_key
        self.base_url = "https://financialmodelingprep.com/stable"
        self.input_file = input_file
        
        # Setup output file
        if output_file is None:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            output_file = f"crypto_history_{timestamp}.csv"
        
        self.output_file = output_file
        self.output_path = f"../outputs/{self.output_file}"
        self.checkpoint_file = f"../outputs/.{output_file}.checkpoint"
        
    def load_crypto_symbols(self) -> List[str]:
        """Load cryptocurrency symbols from the input CSV file"""
        input_path = f"../outputs/{self.input_file}"
        
        if not os.path.exists(input_path):
            print(f"✗ Error: Input file not found at {input_path}")
            return []
        
        try:
            df = pd.read_csv(input_path)
            if 'symbol' not in df.columns:
                print("✗ Error: 'symbol' column not found in input file")
                return []
            
            symbols = df['symbol'].unique().tolist()
            print(f"✓ Loaded {len(symbols)} unique symbols from {self.input_file}")
            return symbols
        except Exception as e:
            print(f"✗ Error reading input file: {e}")
            return []
    
    def fetch_historical_data(self, symbol: str) -> List[Dict]:
        """Fetch 5 years of historical EOD data for a specific symbol"""
        url = f"{self.base_url}/historical-price-eod/light?symbol={symbol}&apikey={self.api_key}"
        
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            
            if data and isinstance(data, list) and len(data) > 0:
                # Filter to last 5 years
                five_years_ago = datetime.now() - timedelta(days=5*365)
                
                filtered_data = []
                for record in data:
                    try:
                        record_date = datetime.strptime(record['date'], '%Y-%m-%d')
                        if record_date >= five_years_ago:
                            filtered_data.append(record)
                    except:
                        # If date parsing fails, include the record anyway
                        filtered_data.append(record)
                
                return filtered_data
            return []
        except Exception as e:
            print(f"✗ Error fetching historical data for {symbol}: {e}")
            return []
    
    def get_already_fetched_symbols(self) -> Set[str]:
        """Get set of symbols that already have data in the output file"""
        if not os.path.exists(self.output_path):
            return set()
        
        try:
            df = pd.read_csv(self.output_path)
            if 'symbol' in df.columns:
                fetched = set(df['symbol'].unique())
                print(f"📂 Found {len(fetched)} symbols with existing historical data")
                return fetched
        except Exception as e:
            print(f"⚠️  Could not read existing file: {e}")
        
        return set()
    
    def write_rows_to_csv(self, rows: List[Dict], is_first: bool = False):
        """Write multiple rows to CSV incrementally"""
        if not rows:
            return
        
        df = pd.DataFrame(rows)
        
        # Write header only for first batch and if file doesn't exist
        write_header = is_first and not os.path.exists(self.output_path)
        
        df.to_csv(self.output_path, mode='a', header=write_header, index=False)
    
    def save_checkpoint(self, processed_symbols: Set[str]):
        """Save checkpoint of processed symbols"""
        with open(self.checkpoint_file, 'w') as f:
            json.dump(list(processed_symbols), f)
    
    def load_checkpoint(self) -> Set[str]:
        """Load checkpoint of processed symbols"""
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'r') as f:
                    return set(json.load(f))
            except:
                pass
        return set()
    
    def fetch_all_historical_data(self, delay: float = 0.2, checkpoint_interval: int = 50,
                                  sample_size: int = None) -> str:
        """
        Fetch 5 years of historical data for all cryptocurrencies with incremental writes
        
        Args:
            delay: Delay between API calls to respect rate limits
            checkpoint_interval: Save checkpoint every N symbols
            sample_size: If specified, only fetch this many symbols (for testing)
        
        Returns:
            Path to output file
        """
        print("=" * 70)
        print("CRYPTOCURRENCY HISTORICAL DATA COLLECTION (5 Years)")
        print("=" * 70)
        
        # Step 1: Load symbols from input file
        print("\n[1/4] Loading cryptocurrency symbols...")
        all_symbols = self.load_crypto_symbols()
        
        if not all_symbols:
            print("No symbols found. Exiting.")
            return None
        
        # Limit sample size if specified
        if sample_size:
            all_symbols = all_symbols[:sample_size]
            print(f"📊 Processing sample of {sample_size} symbols")
        
        # Step 2: Check what's already been fetched
        print(f"\n[2/4] Checking for existing historical data...")
        already_fetched = self.get_already_fetched_symbols()
        
        # Filter out already-fetched symbols
        symbols_to_fetch = [s for s in all_symbols if s not in already_fetched]
        
        if not symbols_to_fetch:
            print("✓ All symbols already have historical data! Nothing to do.")
            return self.output_path
        
        print(f"📋 Need to fetch: {len(symbols_to_fetch)} symbols")
        print(f"✓ Already have: {len(already_fetched)} symbols")
        print(f"🎯 Total: {len(all_symbols)} symbols")
        
        # Step 3: Fetch historical data for each symbol
        print(f"\n[3/4] Fetching and writing historical data incrementally...")
        print("💾 Writing to file as we go...")
        print()
        
        successful_symbols = 0
        failed_symbols = 0
        total_records = 0
        failed_symbol_list = []
        is_first_write = not os.path.exists(self.output_path)
        
        start_time = time.time()
        
        for idx, symbol in enumerate(symbols_to_fetch, 1):
            # Progress indicator with ETA
            if idx % 10 == 0 or idx == len(symbols_to_fetch):
                elapsed = time.time() - start_time
                rate = idx / elapsed if elapsed > 0 else 0
                remaining = len(symbols_to_fetch) - idx
                eta_seconds = remaining / rate if rate > 0 else 0
                eta_str = str(timedelta(seconds=int(eta_seconds)))
                
                print(f"Progress: {idx}/{len(symbols_to_fetch)} ({idx/len(symbols_to_fetch)*100:.1f}%) | "
                      f"Symbols: {successful_symbols} OK, {failed_symbols} Failed | "
                      f"Records: {total_records:,} | ETA: {eta_str}")
            
            # Fetch historical data
            historical_data = self.fetch_historical_data(symbol)
            
            if historical_data:
                # Write immediately to CSV
                self.write_rows_to_csv(historical_data, is_first=is_first_write)
                is_first_write = False
                
                successful_symbols += 1
                total_records += len(historical_data)
                already_fetched.add(symbol)
                
                # Periodic checkpoint
                if idx % checkpoint_interval == 0:
                    self.save_checkpoint(already_fetched)
            else:
                failed_symbols += 1
                failed_symbol_list.append(symbol)
            
            # Rate limiting
            time.sleep(delay)
        
        # Final checkpoint
        self.save_checkpoint(already_fetched)
        
        # Step 4: Summary
        print("\n" + "=" * 70)
        print("COLLECTION SUMMARY")
        print("=" * 70)
        print(f"✓ Successfully fetched: {successful_symbols} symbols")
        print(f"✗ Failed to fetch: {failed_symbols} symbols")
        if failed_symbol_list[:5]:
            print(f"  Examples: {', '.join(failed_symbol_list[:5])}")
        print(f"📊 Total historical records collected: {total_records:,}")
        print(f"💾 Output file: {self.output_file}")
        print(f"📁 Total symbols with data: {len(already_fetched)}")
        print("=" * 70)
        
        return self.output_path


def main():
    # Your API key
    API_KEY = "m8TZJWQFGH7G6x2nowAqKdzDfAyakr0T"
    
    # Input file containing cryptocurrency symbols
    INPUT_FILE = "cryptocurrency_market_data.csv"
    
    # Initialize fetcher
    fetcher = CryptoHistoryFetcher(
        API_KEY,
        input_file=INPUT_FILE,
        output_file="crypto_history_data.csv"
    )
    
    # Fetch historical data
    # For testing, you can use sample_size parameter: sample_size=10
    # For full dataset, remove sample_size or set to None
    filepath = fetcher.fetch_all_historical_data(
        delay=0.2,  # Delay between API calls
        checkpoint_interval=50,  # Save checkpoint every 50 symbols
        sample_size=None  # Set to None for all symbols
    )
    
    if filepath:
        print(f"\n✅ Historical data collection complete!")
        print(f"📁 File location: {filepath}")
        
        # Load and display sample
        df = pd.read_csv(filepath)
        
        print("\n" + "=" * 70)
        print("DATA PREVIEW (First 10 rows)")
        print("=" * 70)
        print(df.head(10).to_string())
        
        print("\n" + "=" * 70)
        print("KEY STATISTICS")
        print("=" * 70)
        print(f"Total records: {len(df):,}")
        print(f"Unique symbols: {df['symbol'].nunique():,}")
        print(f"Date range: {df['date'].min()} to {df['date'].max()}")
        
        if 'volume' in df.columns:
            print(f"Total volume (all records): ${df['volume'].sum():,.0f}")
            print(f"Average daily volume: ${df['volume'].mean():,.0f}")
        
        if 'price' in df.columns:
            print(f"Average price: ${df['price'].mean():,.2f}")
            print(f"Median price: ${df['price'].median():,.2f}")
        
        # Show symbols with most records
        print("\n" + "=" * 70)
        print("TOP 10 SYMBOLS BY RECORD COUNT")
        print("=" * 70)
        top_symbols = df['symbol'].value_counts().head(10)
        for symbol, count in top_symbols.items():
            print(f"{symbol}: {count:,} records")
    else:
        print("\n⚠️  No historical data collected.")


if __name__ == "__main__":
    main()

CRYPTOCURRENCY HISTORICAL DATA COLLECTION (5 Years)

[1/4] Loading cryptocurrency symbols...
✓ Loaded 4771 unique symbols from cryptocurrency_market_data.csv

[2/4] Checking for existing historical data...
📂 Found 4728 symbols with existing historical data
📋 Need to fetch: 43 symbols
✓ Already have: 4728 symbols
🎯 Total: 4771 symbols

[3/4] Fetching and writing historical data incrementally...
💾 Writing to file as we go...

Progress: 10/43 (23.3%) | Symbols: 9 OK, 0 Failed | Records: 12,190 | ETA: 0:00:35
Progress: 20/43 (46.5%) | Symbols: 13 OK, 6 Failed | Records: 17,115 | ETA: 0:00:24
Progress: 30/43 (69.8%) | Symbols: 13 OK, 16 Failed | Records: 17,115 | ETA: 0:00:13
Progress: 40/43 (93.0%) | Symbols: 13 OK, 26 Failed | Records: 17,115 | ETA: 0:00:03
Progress: 43/43 (100.0%) | Symbols: 13 OK, 29 Failed | Records: 17,115 | ETA: 0:00:00

COLLECTION SUMMARY
✓ Successfully fetched: 13 symbols
✗ Failed to fetch: 30 symbols
  Examples: FLOKIDASHUSD, SANINUUSD, KOROMARUUSD, SHAMANUSD, PSY